**Age Model Architecture**

input: image

output: age

regression model

Dataset: UTKFace
Link to download:

part 1: https://drive.usercontent.google.com/download?id=1mb5Z24TsnKI3ygNIlX6ZFiwUj0_PmpAW&export=download&authuser=0&confirm=t&uuid=ed8e8e51-cb3c-4262-8314-d96bb87781d0&at=APcXIO0mPgJPDxNgg9CQHXWo3U2S:1772484352669

part 2: https://drive.usercontent.google.com/download?id=19vdaXVRtkP-nyxz1MYwXiFsh_m_OL72b&export=download&authuser=0&confirm=t&uuid=95a6db6e-c371-4d68-8bd1-f39a6f0eb339&at=APcXIO1iISpyVwxBCULNYhXmO8zf:1772485135426


part 3: https://drive.usercontent.google.com/download?id=1oj9ZWsLV2-k2idoW_nRSrLQLUP3hus3b&export=download&authuser=0&confirm=t&uuid=f8223bbf-436b-4eb6-b67c-914e3f714db2&at=APcXIO1CzF8SjAoIUitQd7krPgcI:1772485183395

### Get the Data

In [ ]:
#!rm -rf datasets/

In [ ]:
# Download and unzip the datasets from UTKface

import tarfile
import urllib.request
import os

download_url_part1 = "https://drive.usercontent.google.com/download?id=1mb5Z24TsnKI3ygNIlX6ZFiwUj0_PmpAW&export=download&authuser=0&confirm=t&uuid=ed8e8e51-cb3c-4262-8314-d96bb87781d0&at=APcXIO0mPgJPDxNgg9CQHXWo3U2S:1772484352669"
download_url_part2 = "https://drive.usercontent.google.com/download?id=19vdaXVRtkP-nyxz1MYwXiFsh_m_OL72b&export=download&authuser=0&confirm=t&uuid=95a6db6e-c371-4d68-8bd1-f39a6f0eb339&at=APcXIO1iISpyVwxBCULNYhXmO8zf:1772485135426"
download_url_part3 = "https://drive.usercontent.google.com/download?id=1oj9ZWsLV2-k2idoW_nRSrLQLUP3hus3b&export=download&authuser=0&confirm=t&uuid=f8223bbf-436b-4eb6-b67c-914e3f714db2&at=APcXIO1CzF8SjAoIUitQd7krPgcI:1772485183395"

# Download the datasets
download_urls = [download_url_part1, download_url_part2, download_url_part3]
face_path = os.path.join("datasets", "faces")
os.makedirs(face_path, exist_ok=True)

for dataset_num in range(len(download_urls)):
  dataset_folder_name = "part" + str(dataset_num + 1)
  if (not os.path.exists(os.path.join(face_path, dataset_folder_name)) and not os.path.exists(os.path.join(face_path, "images"))):
    print(f"Extracting dataset {dataset_num+1}...")
    faces_tar_location = os.path.join(face_path, "faces" + str(dataset_num+1) + ".tgz")
    urllib.request.urlretrieve(download_urls[dataset_num], faces_tar_location)

    # extract tar dataset
    faces_tgz = tarfile.open(faces_tar_location)
    faces_tgz.extractall(path="datasets/faces")
    faces_tgz.close()

    # remove tar file
    os.remove(faces_tar_location)

  else:
    print(f"dataset {dataset_num+1} already exists")



In [ ]:
# Combine all data into one folder
import shutil

faces_images_path = os.path.join(face_path, "images")
os.makedirs(faces_images_path, exist_ok=True)

# Get source folders to copy image data from
source_folders = []
for dataset in range(len(download_urls)):
  dataset_path = os.path.join("./datasets/faces/part" + str(dataset+1))
  if (os.path.exists(dataset_path)):
    source_folders.append(dataset_path)

for folder in source_folders:
  file_names = os.listdir(folder)
  for file_name in file_names:
    shutil.move(os.path.join(folder, file_name), faces_images_path)
  os.rmdir(folder) # remove directory because we don't need it anymore

In [ ]:
!pip3 install pillow rich rich-pixels
!pip3 install setuptools==81.0
!pip3 install face-recognition
!pip3 install git+https://github.com/ageitgey/face_recognition_models

In [ ]:
file_names = os.listdir(faces_images_path)
print(f"Number of images: {len(file_names)}")

### Cropping Images & Adding Padding

In [ ]:
from concurrent.futures import ProcessPoolExecutor, as_completed
from face_processor import process_single_image

os.makedirs(os.path.join(face_path, "cropped"), exist_ok=True)
test_set = file_names # test batch

results = []
futures = []
num_workers = os.cpu_count() // 2
images_already_processed = os.listdir("./datasets/faces/cropped")

with ProcessPoolExecutor(max_workers=num_workers) as ppe:
  for img in test_set:
    if img not in images_already_processed:
      submission = ppe.submit(process_single_image, img)
      futures.append(submission)
  for fut in as_completed(futures):
    results.append(fut.result())

num_success = 0
for fn, success_boolean, msg in results:
  if success_boolean:
    num_success += 1

print(f"Processed {num_success}/{len(test_set)} images successfully")

In [ ]:
import random

# Shuffling image data before splitting it

random.seed(49328042)
images_shuffled = file_names
random.shuffle(images_shuffled)

age_labels = []

for file in file_names:
    age_label = int(file[0:file.index('_')])
    age_labels.append(age_label)


In [ ]:
# Split the data into training, validation, and testing sets

train_data = images_shuffled[:16653]
train_labels = age_labels[:16653]

validation_data = images_shuffled[16653:19053]
validation_labels = age_labels[16653:19053]

testing_data = images_shuffled[19053:]
testing_labels = age_labels[19053:]


In [ ]:
# Create image set directories based on labels

import os, shutil, pathlib

original_dir = "./" + str(pathlib.Path("datasets/faces/cropped"))
base_training_dir = "./" + str(pathlib.Path("datasets/faces/Training"))

def make_data_sets(subdir, starting_index, ending_index):
    for img_index in range(starting_index, ending_index):
        dir_path = f"{base_training_dir}/{subdir}/{str(age_labels[img_index])}"
        os.makedirs(dir_path, exist_ok=True)

        image_path = os.path.join(original_dir, images_shuffled[img_index])
        if (image_path not in os.listdir(dir_path)):
            shutil.move(image_path, dir_path)

make_data_sets("validation", 16653, 19053)



Error: Destination path './datasets/faces/Training/validation/29/29_0_3_20170119195309387.jpg' already exists

### Building the CNN

In [ ]:
%pip install tensorflow
%pip install numpy

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

# 1 Input Layer
# 5 Convolutional Layers
# 4 Max Pooling Layers
# 1 Output Layer (after flattening)

inputs = keras.Input(shape=(160, 160, 3))
x = layers.Rescaling(1./255)(inputs)
x = layers.Conv2D(filters=32, kernel_size=3, activation="relu")(x)
x = layers.MaxPooling2D(pool_size=2)(x)
x = layers.Conv2D(filters=64, kernel_size=3, activation="relu")(x)
x = layers.MaxPooling2D(pool_size=2)(x)
x = layers.Conv2D(filters=128, kernel_size=3, activation="relu")(x)
x = layers.MaxPooling2D(pool_size=2)(x)
x = layers.Conv2D(filters=256, kernel_size=3, activation="relu")(x)
x = layers.MaxPooling2D(pool_size=2)(x)
x = layers.Conv2D(filters=256, kernel_size=3, activation="relu")(x)
x = layers.Flatten()(x)
outputs = layers.Dense(1)(x)

model = keras.Model(inputs=inputs, outputs=outputs)

model.summary()
